<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-13/AI_Modul_13.3_Praktikum_Pengembangan_Aplikasi_Sederhana.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 13.3: Praktikum Pengembangan Aplikasi Visi Komputer Sederhana
**Tujuan Pembelajaran:**
1. Membangun generator streaming video web berbasis protokol HTTP Multipart MJPEG.
2. Merancang arsitektur thread-safe pemisahan antara mesin komputasi AI dan antarmuka pengguna.
3. Mensimulasikan dasbor inspeksi mutu industri kelapa sawit real-time.

In [ ]:
import cv2
import time
import threading
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"OpenCV Siap: {cv2.__version__}")

### Bagian 1: Mesin Pembuat Stream MJPEG Asinkron

In [ ]:
class MockStreamEngine:
    def __init__(self, width=480, height=360):
        self.width = width
        self.height = height
        self.lock = threading.Lock()
        self.current_jpeg = None
        self.running = False
        self.worker = None
        self.frame_id = 0
        
    def start(self):
        self.running = True
        self.worker = threading.Thread(target=self._run, daemon=True)
        self.worker.start()
        return self
        
    def _run(self):
        while self.running:
            self.frame_id += 1
            canvas = np.full((self.height, self.width, 3), 45, dtype=np.uint8)
            
            # Animasi konveyor
            x = int((self.frame_id * 6) % self.width)
            cv2.circle(canvas, (x, self.height // 2), 35, (10, 140, 240), -1) # Buah Matang BGR
            cv2.rectangle(canvas, (x-40, self.height//2-40), (x+40, self.height//2+40), (0, 255, 0), 2)
            
            # Label Dasbor
            cv2.putText(canvas, "PORTAL INSPEKSI MUTU SAWIT", (15, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(canvas, f"FRAME: #{self.frame_id} | STATUS: NORMAL", (15, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            ret, enc = cv2.imencode('.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 75])
            if ret:
                with self.lock:
                    self.current_jpeg = enc.tobytes()
            time.sleep(0.03)
            
    def get_latest_jpeg(self):
        with self.lock:
            return self.current_jpeg
            
    def stop(self):
        self.running = False
        if self.worker is not None:
            self.worker.join(timeout=1.0)

server = MockStreamEngine().start()
time.sleep(0.2)
sample_bytes = server.get_latest_jpeg()
print(f"Ukuran payload JPEG streaming: {len(sample_bytes):,} bytes")
assert sample_bytes is not None and len(sample_bytes) > 0, "Gagal mengompresi JPEG!"
print("[VALIDASI SUKSES] Generator stream MJPEG aktif dan memproduksi buffer terenkripsi!")

### Bagian 2: Simulasi Generator Response HTTP Multipart

In [ ]:
def simulate_http_mjpeg_stream(stream_hub, num_frames=5):
    chunks = []
    for _ in range(num_frames):
        jpg = stream_hub.get_latest_jpeg()
        if jpg is None:
            time.sleep(0.02)
            continue
        # Format standar RFC multipart/x-mixed-replace
        part_header = b"--frame\r\nContent-Type: image/jpeg\r\nContent-Length: " + str(len(jpg)).encode() + b"\r\n\r\n"
        chunk = part_header + jpg + b"\r\n"
        chunks.append(chunk)
        time.sleep(0.03)
    return chunks

stream_chunks = simulate_http_mjpeg_stream(server, num_frames=5)
server.stop()
print(f"Berhasil memproduksi {len(stream_chunks)} blok paket transmisi HTTP Multipart!")
print(f"Contoh Header Paket: {stream_chunks[0][:75]}")

### Bagian 3: Dekode Citra Terakhir dan Verifikasi Dasbor

In [ ]:
# Rekonstruksi citra dari byte stream terakhir
nparr = np.frombuffer(sample_bytes, np.uint8)
decoded_img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(decoded_img, cv2.COLOR_BGR2RGB))
plt.title('Tampilan Dasbor Aplikasi Web Inspeksi Mutu Sawit (MJPEG)')
plt.axis('off')
plt.tight_layout()
plt.savefig('praktikum_aplikasi_dasbor_mjpeg_13_3.png', dpi=200)
plt.close()
print("[OK] Tangkapan layar dasbor tersimpan di 'praktikum_aplikasi_dasbor_mjpeg_13_3.png'")